# Bronze Delta Processing Overview
This notebook authenticates to ADLS, reads raw retail source data from the bronze layer, adds ingestion audit columns, and writes standardized Delta datasets for downstream silver processing.

In [0]:
# Purpose: Configure ADLS OAuth access and load the bronze orders dataset to validate the raw source can be read.
# -- ADLS Gen2 OAuth (Service Principal) auth --
storage_account = "silveradlsstorage"
scope = "retail-adls-kv-scope"

spark.conf.set(
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net",
    "OAuth"
)
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net",
    dbutils.secrets.get(scope=scope, key="adls-sp-client-id")
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net",
    dbutils.secrets.get(scope=scope, key="adls-sp-client-secret")
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{dbutils.secrets.get(scope=scope, key='adls-sp-tenant-id')}/oauth2/token"
)

# -- Load orders from Bronze --
orders_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/retail/orders/"

orders_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .format('delta').load(orders_path)
)

display(orders_df)


In [0]:
# Purpose: Enrich the orders dataset with batch metadata and audit columns before writing it as Delta.
from pyspark.sql import functions as F
batch_id = "manual-20260602-001"
source_system = "pos"
entity_name = "orders"
load_date = "2026-06-01"
orders_bronze_df = (
    orders_df
    .withColumn("BatchId", F.lit(batch_id))
    .withColumn("SourceSystem", F.lit(source_system))
    .withColumn("EntityName", F.lit(entity_name))
    .withColumn("LoadDate", F.lit(load_date))
    .withColumn("IngestedAtUtc", F.current_timestamp())
    .withColumn("InputFileName", F.col("_metadata.file_path"))
)


In [0]:
# Purpose: Persist the enriched orders dataset into the bronze Delta zone for downstream consumption.
orders_delta_path = "abfss://bronze@silveradlsstorage.dfs.core.windows.net/retail_delta/orders"
(
    orders_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(orders_delta_path)
)



In [0]:
# Purpose: Read back the written bronze Delta orders dataset to verify the load succeeded.
orders_delta_df = spark.read.format("delta").load(orders_delta_path)
display(orders_delta_df)




In [0]:
# Purpose: Set shared runtime variables used to process multiple bronze source entities in a reusable loop.
from pyspark.sql import functions as F
storage_account_name = "silveradlsstorage"
load_date = "2026-06-01"
batch_id = "manual-20260602-001"
bronze_base = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net"



In [0]:
# Purpose: Create a reusable helper that appends standard bronze audit metadata and a row-level hash to any source dataset.
def add_bronze_audit_columns(df, source_system, entity_name, batch_id, load_date):
    original_columns = df.columns
    record_hash_expr = F.sha2(
        F.concat_ws(
            "||",
            *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in original_columns]
        ),
        256
    )
    return df.withColumns({
        "BatchId": F.lit(batch_id),
        "SourceSystem": F.lit(source_system),
        "EntityName": F.lit(entity_name),
        "LoadDate": F.lit(load_date),
        "IngestedAtUtc": F.current_timestamp(),
        "InputFileName": F.col("_metadata.file_path"),
        "RecordHash": record_hash_expr,
    })


In [0]:
# Purpose: Define the bronze source-to-target mapping for every retail entity that must be landed as Delta.
entities = [
    {
        "source_system": "pos",
        "entity_name": "orders",
        "format": "csv",
        "source_path": f"{bronze_base}/retail/pos/orders/load_date={load_date}/orders.csv",
        "delta_path": f"{bronze_base}/retail_delta/orders"
    },
    {
        "source_system": "pos",
        "entity_name": "order_items",
        "format": "csv",
        "source_path": f"{bronze_base}/retail/pos/order_items/load_date={load_date}/order_items.csv",
        "delta_path": f"{bronze_base}/retail_delta/order_items"
    },
    {
        "source_system": "crm",
        "entity_name": "customers",
        "format": "csv",
        "source_path": f"{bronze_base}/retail/crm/customers/load_date={load_date}/customers.csv",
        "delta_path": f"{bronze_base}/retail_delta/customers"
    },
    {
        "source_system": "catalog",
        "entity_name": "products",
        "format": "csv",
        "source_path": f"{bronze_base}/retail/catalog/products/load_date={load_date}/products.csv",
        "delta_path": f"{bronze_base}/retail_delta/products"
    },
    {
        "source_system": "inventory",
        "entity_name": "inventory",
        "format": "csv",
        "source_path": f"{bronze_base}/retail/inventory/inventory/load_date={load_date}/inventory.csv",
        "delta_path": f"{bronze_base}/retail_delta/inventory"
    },
    {
        "source_system": "payments",
        "entity_name": "payments",
        "format": "csv",
        "source_path": f"{bronze_base}/retail/payments/payments/load_date={load_date}/payments.csv",
        "delta_path": f"{bronze_base}/retail_delta/payments"
    },
    {
        "source_system": "stores",
        "entity_name": "stores",
        "format": "csv",
        "source_path": f"{bronze_base}/retail/stores/stores/load_date={load_date}/stores.csv",
        "delta_path": f"{bronze_base}/retail_delta/stores"
    },
    {
        "source_system": "payments",
        "entity_name": "payment_methods",
        "format": "json",
        "source_path": f"{bronze_base}/retail/payments/payment_methods/load_date={load_date}/payment_methods.json",
        "delta_path": f"{bronze_base}/retail_delta/payment_methods"
    }
]



In [0]:
# Purpose: Loop through every configured retail entity, read the source file, add audit columns, and write the bronze Delta output.
for entity in entities:
    print(f"Processing {entity['entity_name']}")
    if entity["format"] == "csv":
        df = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .csv(entity["source_path"])
        )
    elif entity["format"] == "json":
        df = (
            spark.read
            .option("multiLine", "true")
            .json(entity["source_path"])
        )
    else:
        raise ValueError(f"Unsupported format: {entity['format']}")
    bronze_df = add_bronze_audit_columns(
        df=df,
        source_system=entity["source_system"],
        entity_name=entity["entity_name"],
        batch_id=batch_id,
        load_date=load_date
    )
    (
        bronze_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(entity["delta_path"])
    )
    print(f"Completed {entity['entity_name']} -> {entity['delta_path']}")



In [0]:
# Purpose: Verify each bronze Delta table was created successfully by checking row counts.
for entity in entities:
    delta_df = spark.read.format("delta").load(entity["delta_path"])
    print(entity["entity_name"], delta_df.count())

